# Kombinierte Labels: HistGradientBoosting

**Hypothese:** Additive Splits verbessern die Trennung der 23 kombinierten Crop/Stage-Klassen gegenüber Extra Trees und erreichen mindestens die kombinierte BAcc der RBF-SVM.

Die Suche nutzt ausschließlich die fünf gemeinsamen Trainings-Folds. Der Validierungsanteil wird genau einmal nach Auswahl der besten CV-Konfiguration bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_hist_gradient_boosting
from awp2.preprocessing import PreprocessingConfig

COMBINED_EXTRA_TREES_CV_BACC = 0.81228
COMBINED_EXTRA_TREES_VAL_BACC = 0.8039
COMBINED_SVM_CV_BACC = 0.834
COMBINED_SVM_VAL_BACC = 0.845

hist_gradient_boosting_preprocessing = PreprocessingConfig(scale=False)

## 1. Hyperparameter suchen

HistGradientBoosting besitzt keine Klassengewichte. `balance_samples=True` gewichtet daher jede kombinierte Klasse im Training gleich. Die acht Kandidaten variieren Lernrate, maximale Blattanzahl und L2-Regularisierung; die gemeinsame Vorverarbeitung bleibt unskaliert und verwendet AEZ sowie Month.

In [ ]:
tuned = tune(
    make_combined_hist_gradient_boosting(),
    TuneConfig(
        name="combined_hist_gradient_boosting_search",
        approach="combined",
        description=(
            "HistGradientBoosting auf kombinierten Crop/Stage-Labels: "
            "Lernrate, Blattanzahl und L2-Regularisierung vergleichen."
        ),
        preprocessing=hist_gradient_boosting_preprocessing,
        balance_samples=True,
        param_grid={
            "estimator__learning_rate": [0.05, 0.1],
            "estimator__max_leaf_nodes": [15, 31],
            "estimator__l2_regularization": [0.0, 1.0],
        },
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## 2. Einmal validieren

Die beste Konfiguration aus der Trainings-CV wird einmal auf dem gemeinsamen Validierungsanteil bewertet. Das Ergebnis ordnet HistGradientBoosting gegenüber Extra Trees und der RBF-SVM ein; es verändert weder die Suchmenge noch die Auswahl des Kandidaten.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_hist_gradient_boosting",
        approach="combined",
        description=(
            "Beste HistGradientBoosting-Konfiguration aus "
            "combined_hist_gradient_boosting_search, einmal auf der Validierung bewertet."
        ),
        preprocessing=hist_gradient_boosting_preprocessing,
        balance_samples=True,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. Analyse

Die Vergleichstabelle trennt CV und Validierung. Die Abbildungen zeigen Kandidatenstreuung, Klassenrecall und die häufigsten kombinierten Verwechslungen.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = pd.DataFrame(
    {
        "CV-BAcc kombiniert": {
            "HistGradientBoosting": tuned.best_score,
            "Extra Trees": COMBINED_EXTRA_TREES_CV_BACC,
            "RBF-SVM": COMBINED_SVM_CV_BACC,
        },
        "Validierungs-BAcc kombiniert": {
            "HistGradientBoosting": result.metrics.bacc_combined,
            "Extra Trees": COMBINED_EXTRA_TREES_VAL_BACC,
            "RBF-SVM": COMBINED_SVM_VAL_BACC,
        },
    }
)
display(comparison)

top_confusions = (
    pd.DataFrame(
        {"Wahr": combined_label(result.y_val), "Vorhergesagt": combined_label(result.y_pred)}
    )
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
display(top_confusions)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__learning_rate",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: CV-Score von HistGradientBoosting")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(
    tuning_ax.figure,
    "combined_hist_gradient_boosting_tuning",
    MODEL_DOCS_FIGURES_DIR,
)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(
    confusion_figure,
    "combined_hist_gradient_boosting_confusion",
    MODEL_DOCS_FIGURES_DIR,
)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(
    recall_figure,
    "combined_hist_gradient_boosting_recall",
    MODEL_DOCS_FIGURES_DIR,
)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(
    confusions_figure,
    "combined_hist_gradient_boosting_confusions",
    MODEL_DOCS_FIGURES_DIR,
)

{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "invalid_combinations": result.metrics.invalid_combinations,
}